In [1]:
!pip install psycopg2-binary boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.2 MB/s eta 0:00:00


In [3]:
import boto3
import psycopg2

# 1. Configurações de Acesso
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
AWS_REGION = "us-east-1"
BUCKET_NAME = "s3-postsgres-data-catalog"
PREFIX = "imagens/"

DB_HOST = ""
DB_PORT = "5432"
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = ""

try:
    # 2. Conectar ao PostgreSQL
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    cursor = conn.cursor()
    print("Conectado ao PostgreSQL com sucesso!")

    # 3. Criar tabela de inventário (apenas ID e Nome do Arquivo)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS catalogo_imagens (
            id SERIAL PRIMARY KEY,
            nome_arquivo VARCHAR(255) NOT NULL
        );
    """)
    conn.commit()

    # 4. Conectar ao S3 e listar objetos
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        region_name=AWS_REGION
    )

    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX)

    # 5. Inserir nomes dos arquivos no banco
    if 'Contents' in response:
        for obj in response['Contents']:
            chave = obj['Key']
            if chave != PREFIX and not chave.endswith('/'):
                nome_arquivo = chave.split('/')[-1]
                cursor.execute(
                    "INSERT INTO catalogo_imagens (nome_arquivo) VALUES (%s);",
                    (nome_arquivo,)
                )
                print(f"Catalogado: {nome_arquivo}")
        conn.commit()

    # 6. Consultar e exibir o inventário
    cursor.execute("SELECT * FROM catalogo_imagens;")
    for linha in cursor.fetchall():
        print(linha)

except Exception as e:
    print(f"Erro: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn:
        conn.close()
    print("Conexões encerradas.")

Conectado ao PostgreSQL com sucesso!
Catalogado: imagem1.jpg
Catalogado: imagem2.jpg
(1, 'imagem1.jpg')
(2, 'imagem2.jpg')
Conexões encerradas.
